# Build a Chatbot, Step by Step

In this notebook, you'll build an AI chatbot piece by piece. Each cell adds one concept — run them in order, and by the end you'll understand every part of the terminal chatbot in `python/chat.py`.

**What you'll learn:**
- Connecting to Azure AI Foundry using the OpenAI SDK
- Sending a message and getting a response (the Responses API)
- Multi-turn conversation using `previous_response_id`
- Changing the chatbot's personality with system instructions

**Prerequisites:** Make sure you've created your `.env` file with your Azure API key and endpoint. See the [root README](../README.md#quick-setup) for setup instructions.

## Step 1: Import Libraries

We only need three things:
- **`openai`** — The official OpenAI Python SDK. It also works with Azure AI Foundry since Azure exposes an OpenAI-compatible API.
- **`dotenv`** — Loads configuration from a `.env` file so we don't hard-code secrets in our code.
- **`os` / `Path`** — Standard library helpers for reading environment variables and building file paths.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

print("Libraries imported!")

## Step 2: Load Environment Variables

Your API key, endpoint URL, and other settings live in a `.env` file at the project root. We load them here so we never hard-code secrets into our source code.

The key settings are:
- **`AZURE_OPENAI_ENDPOINT`** — The URL of your Azure AI Foundry deployment
- **`AZURE_OPENAI_API_KEY`** — Your secret API key
- **`MODEL_NAME`** — Which model to use (defaults to `gpt-5.2`)
- **`REASONING_EFFORT`** — How hard the model thinks: `low`, `medium`, or `high`
- **`CHATBOT_INSTRUCTIONS`** — The system prompt that sets the chatbot's personality

In [ ]:
# The .env file lives at the project root (one level up from this notebook)
env_path = Path.cwd().parent / ".env"

# If running from somewhere unexpected, try the standard location
if not env_path.exists():
    env_path = Path.cwd() / ".env"

load_dotenv(env_path)

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
MODEL = os.getenv("MODEL_NAME", "gpt-5.2")
REASONING_EFFORT = os.getenv("REASONING_EFFORT", "low")
INSTRUCTIONS = os.getenv(
    "CHATBOT_INSTRUCTIONS",
    "You are a helpful assistant. Be concise and friendly.",
)

# Quick check that the essentials are set
if not ENDPOINT or not API_KEY:
    print("ERROR: Missing configuration!")
    print(f"  Looked for .env at: {env_path}")
    print("  Make sure AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY are set.")
else:
    print(f"Configuration loaded!")
    print(f"  Endpoint: {ENDPOINT[:40]}...")
    print(f"  Model:    {MODEL}")
    print(f"  Effort:   {REASONING_EFFORT}")

## Step 3: Create the OpenAI Client

Azure AI Foundry exposes an **OpenAI-compatible API**. That means we can use the exact same `OpenAI` SDK we'd use with OpenAI directly — we just point `base_url` at our Azure endpoint instead.

In [ ]:
client = OpenAI(
    base_url=ENDPOINT,
    api_key=API_KEY,
    max_retries=10,
)

print("Client created! Ready to chat.")

## Step 4: Send Your First Message

This is the core of the chatbot — a single call to `client.responses.create()`. Let's break down each parameter:

| Parameter | What it does |
|-----------|-------------|
| `model` | Which AI model to use |
| `input` | The user's message (a string) |
| `instructions` | System-level instructions that set the chatbot's personality |
| `reasoning` | Controls how much "thinking" the model does before answering |

Run this cell to send a message and see the AI's response!

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="What is machine learning in one sentence?",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
)

print(f"Assistant: {response.output_text}")
print()

# The API also tells us how many tokens were used
if response.usage:
    print(f"Tokens used: {response.usage.input_tokens} in, {response.usage.output_tokens} out, {response.usage.total_tokens} total")

## Step 5: Multi-Turn Conversation

So far we've sent one message and gotten one response. But a real chatbot remembers what you said before.

The Responses API makes this easy with **`previous_response_id`**. The API stores the full conversation history on the server — you just pass back the ID from the last response, and the API knows which conversation to continue.

```
Turn 1:  You send "What is Python?"          → API returns response (id: resp_abc)
Turn 2:  You send "What are its main uses?"   → API sees previous_response_id=resp_abc
         + previous_response_id=resp_abc        and knows "its" means Python
```

Let's try it — we'll ask a question, then ask a follow-up that only makes sense if the model remembers the first message.

In [ ]:
# --- Turn 1: Ask a question ---
response1 = client.responses.create(
    model=MODEL,
    input="What is Python?",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
)

print(f"You:       What is Python?")
print(f"Assistant: {response1.output_text}")
print(f"\n--- Response ID: {response1.id} ---\n")

# --- Turn 2: Ask a follow-up using previous_response_id ---
# Notice we say "its" without specifying Python — the model remembers!
response2 = client.responses.create(
    model=MODEL,
    input="What are its main uses?",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    previous_response_id=response1.id,  # <-- This links the conversation!
)

print(f"You:       What are its main uses?")
print(f"Assistant: {response2.output_text}")

## Step 6: Try Different Instructions

The `instructions` parameter is like giving the chatbot a personality. Try changing the instructions below and re-running the cell to see how the response changes.

Some fun ones to try:
- `"You are a pirate who teaches coding. Say 'Arrr' frequently."`
- `"Explain everything as if the user is 5 years old."`
- `"You must respond entirely in haiku format (5-7-5 syllable structure)."`
- `"You review code like Gordon Ramsay reviews food. Be dramatic and brutally honest."`

In [ ]:
# Change these and re-run the cell!
custom_instructions = "You are a pirate who teaches coding. Say 'Arrr' frequently."
custom_message = "What is a for loop?"

response = client.responses.create(
    model=MODEL,
    input=custom_message,
    instructions=custom_instructions,
    reasoning={"effort": REASONING_EFFORT},
)

print(f"Instructions: {custom_instructions}")
print(f"You:          {custom_message}")
print(f"Assistant:    {response.output_text}")

## You Built a Chatbot!

You've now used every piece that makes up the terminal chatbot:

1. **Import & configure** — load your API credentials
2. **Create a client** — point the OpenAI SDK at Azure
3. **Send a message** — `client.responses.create()`
4. **Chain a conversation** — pass `previous_response_id` to maintain context
5. **Customize personality** — change `instructions` to alter behavior

The full terminal chatbot in `python/chat.py` wraps all of this in a `while True` loop that reads your input, sends it to the API, and prints the response. That's it — you already understand every piece!

### Try it out

Run the full chatbot from a terminal:
```bash
cd level-1-chat/python
python chat.py
```

### What's next?

- **[Level 1B — Streaming](../level-1-chat-streaming/)** — see tokens appear one by one as the AI generates them
- **[Level 2 — File Upload](../level-2-files/)** — attach images and documents to your messages
- **[Level 3 — Web UI](../level-3-web/)** — build a browser-based chat interface
- **[PRDs](../PRDs/)** — workshop exercises to extend your chatbot with Claude Code